# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [ ]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-01','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** _..._

The one row of the table that I am building will represent one line item of each order, with the item name, order number, vendor ID, and qty.

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [ ]:


# TODO: pd.json_normalize(orders_json, record_path='lines',, meta=[...])
# TODO: assert the row count and the quantity total

import pandas as pd

lines = pd.json_normalize(
    orders_json,
    record_path = 'lines',
    meta = ['order', 'vendor_id']
)

assert len(lines) == 4
assert lines['qty'].sum() == 7

lines

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-01


I used pd.json_normalize to make the nested data into a table and record_path = 'lines' makes each item of the originial data its own row. The new table displaying items from each tabel is named 'lines'.

### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [ ]:
# TODO
vendors = pd.json_normalize(vendors_json)

joined = lines.merge(
    vendors,
    on = 'vendor_id',
    how = 'left',
    indicator = True
)

unmatched = (
    joined.loc[joined['_merge'] == 'left_only']
    .groupby('vendor_id', as_index = False)['qty']
    .sum()
)

display(joined)
display(unmatched)


joined.isna().sum()
joined.loc[joined['_merge'] == 'left_only']

,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-01,Hoos Burgers,A,both


,vendor_id,qty


,item,qty,order,vendor_id,name,zone,_merge


I used pd.json_normalize to convert the vendors list into a table. I matched each item row to the vendor using the vendor_id. I kept every row from the new 'lines' table using a left join. Unmatched was a precaution in case there was an item which did not match a vendor. I ran the join checks at the bottom of the cell to make sure I did not fail to notice a missing value.

I think the missing values are not showing because all of the json payload has all of the vendors accounted for. I cannot find the missing values.

### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [ ]:
# TODO

vendor_totals = (
    joined.assign(name = joined['name'].fillna('Unknown vendor'))
    .groupby('name', as_index = False)['qty']
    .sum()
)

assert vendor_totals['qty'].sum() == joined['qty'].sum()
display(vendor_totals)



,name,qty
0,Cav Merch,1
1,Hoos Burgers,6


Ran vendor_total to find the qty of items from each vendor. The total qty is 7, which is the total amount of items from the payload. I ran an assert to confirm that the .sum was equal to the qty joined.

### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [ ]:
# TODO

zone_totals = (
    joined.groupby('zone', as_index=False)['qty']
    .sum()
    .sort_values('qty', ascending=False)

)

known_zone_units = zone_totals['qty'].sum()
missing_zone_units = joined.loc[joined['zone'].isna(), 'qty'].sum()

display(zone_totals)
print(f'Known zone units: {known_zone_units}')
print(f'Missing zone units: {missing_zone_units}')

assert known_zone_units + missing_zone_units == joined['qty'].sum()

,zone,qty
0,A,7


Known zone units: 7
Missing zone units: 0


I ran groupby 'zone', summing the qty, to find the total qty of items for each zone. The known_zone_units added all the qty to make sure the total for all zones was 7. The missing_zone_units checked to make sure that no zones were missing.

### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [ ]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# TODO

prepared_order = [
    {**order, 'lines': order.get('lines', [])}
    for order in ragged_json
]

ragged_lines = pd.json_normalize(
    prepared_order,
    record_path = 'lines',
    meta = ['order', 'vendor_id']
)

ragged_lines['qty'] = ragged_lines['qty'].astype('Int64')

display(ragged_lines)

assert len(ragged_lines) == 2
assert ragged_lines['qty'].isna().sum() == 1

,item,qty,order,vendor_id
0,Soda,2,4,V-01
1,Fries,<NA>,6,V-01


I first prepared the new orders creating a blank 'lines' so the normalization wouldn't running through order 5. I copied the orders in a new dictionary and ran a for loop to go through each. After this I flatted the the items and made a table to show 'order' and 'vendor_id, and lastly changed the data type to 'Int64' to hold on to the missing qty from order 6. The asserts checked there were two rows and that there is one missing qty.

### Q6 — Validate

In [ ]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


ALl the checks passed first try. This is the first time this has happened!

### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a)
One row per line item is a useful shape because it makes easy to visualize each item and qty in individual rows. This item info can be seen while the vendor id and order number are attached. This makes it really easy to answer the questions asking how much qty of each item were sold. This would be difficult to answer with one row per order because some orders have multiple items that would have to sorted through in orginal nested formate.

b)
A quantity of zero counts toward calculations and is stored as integer. Compared to missing quantity that simply tells python the value is unknown. This means that downstream, calculations would break because the zero makes it seem as if the quantity is zero and not unknown. Someone else viewing the code would then think the table is complete and not realize there was data missing.